# Stage 1 — GMM + EM (unsupervised lesion segmentation)

Fit a Gaussian Mixture Model in **HSV** color space to cluster leaf pixels into **rust / green / background**, then visualize the lesion overlay (the core PhytoLabs UX).

If you don't have a real dataset yet, this notebook generates a small synthetic one so everything runs end-to-end.

In [ ]:
from _setup import DATA_DIR, ensure_dataset
import numpy as np
import matplotlib.pyplot as plt
from phytolabs import io, segmentation, viz

data_dir = ensure_dataset()

## Load training images

In [ ]:
healthy_imgs, healthy_paths = io.load_folder(data_dir / 'train' / 'healthy')
rust_imgs, rust_paths = io.load_folder(data_dir / 'train' / 'rust')
all_imgs = healthy_imgs + rust_imgs
print(f'{len(healthy_imgs)} healthy, {len(rust_imgs)} rust training images')

plt.imshow(viz.bgr_to_rgb(rust_imgs[0])); plt.axis('off'); plt.title('example rust leaf');

## Fit the GMM

We pool subsampled HSV pixels across all training images and fit one global GMM, then label each component by its mean hue/saturation.

In [ ]:
leaf_gmm = segmentation.train_gmm(all_imgs, k=4, random_state=0)
for c in range(leaf_gmm.n_components):
    h, s, v = leaf_gmm.gmm.means_[c]
    print(f'component {c}: HSV mean=({h:.0f},{s:.0f},{v:.0f}) -> {leaf_gmm.label_map[c]}')

## Visualize segmentation + lesion overlay

Original, lesion overlay, rust mask, and green mask for a diseased leaf.

In [ ]:
bgr = rust_imgs[0]
seg = leaf_gmm.segment(bgr)
viz.plot_segmentation(bgr, seg, title='Rust leaf segmentation')
plt.show()

In [ ]:
# Same for a healthy leaf — expect (almost) no rust pixels.
bgr_h = healthy_imgs[0]
viz.plot_segmentation(bgr_h, leaf_gmm.segment(bgr_h), title='Healthy leaf segmentation')
plt.show()

## Persist the model

Saved as `artifacts/gmm.joblib` for the later stages / CLI.

In [ ]:
from _setup import ARTIFACTS_DIR
leaf_gmm.save(ARTIFACTS_DIR / 'gmm.joblib')
print('saved', ARTIFACTS_DIR / 'gmm.joblib')

> **Tuning on real data:** inspect the printed component HSV means. If rust tissue is mislabeled, adjust the hue ranges in `phytolabs.segmentation.classify_component`.